# Recovery: rebuild cross-validation scores from saved models

**Purpose.** `my_classifiers` (the per-fold accuracies and predictions returned by
`experiment.cross_validation_experiment`) lives only in memory. If the kernel is
restarted before the scores are saved, they are lost, but the trained models in
`Project1/results/<system>/seed-<seed>/fold-<k>/model*` are still on disk.
This notebook recomputes the scores by loading each saved model and re-scoring it on
the held-out fold it was not trained on, then stores them with `save_system`.

**Use it only for recovery.** It does not train anything and does not modify the model
files. For normal work, train with `run_system(...)` in the main notebook, which saves
scores automatically, and analyse with `my_classifiers = load_all_systems()`.

**Requirements (must match the original training exactly).**
- `my_seeds` and `num_folds` (the cross-validation splits are regenerated from these)
- the same tokenizer file (`Project1/results/bpe_8000.json`) and the same encoder
  classes (`TokenIndicator`, `BagOfWords`)
- the saved model folders present under `Project1/results/`

**Output.** One file per system in `Project1/results/stored_results/<system>.pkl`.

**Run order.** 1) setup cells (imports, dataset, tokenizer, seeds, splits, encoders,
`save_system`), 2) `rebuild_system` definition, 3) the loop over `systems`.

**Notes.**
- Slow: encoding about 13,000 sentences per fold in pure Python takes several minutes
  per system.
- Sanity check: the rebuilt MLP accuracies should match the best-epoch validation
  accuracies printed during the original training.
- To recover only some systems, edit the `systems` list.

In [9]:
import os
if os.path.basename(os.getcwd()) == "Project1":
    os.chdir("..")
print(os.getcwd())

import pickle
import numpy as np
from collections import OrderedDict
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from tokenizers import Tokenizer

import Project1.experiment as experiment
import Project1.clslib as clslib
import Project1.loglin as loglin
import Project1.smlp as smlp

C:\My documents\UvA\NLP\Projects\Project_1\git\NLP


In [11]:
def rebuild_system(name, encoder, kind, seeds=None):
    """kind: 'logistic' or 'mlp'. Re-scores saved fold models on their held-out folds."""
    seeds = seeds if seeds is not None else my_seeds
    runs = []
    for run, seed in enumerate(seeds):
        splits = my_cross_validation_sets[run]
        models, predictions, accuracies = OrderedDict(), OrderedDict(), OrderedDict()
        for fold in splits:
            path = f"Project1/results/{name}/seed-{seed}/fold-{fold}/model"
            texts = [t for t, _ in splits[fold]["validation"]]
            gold  = [l for _, l in splits[fold]["validation"]]
            if kind == 'logistic':
                preds = loglin.predict(texts, loglin.load(path), encoder)
            else:
                preds = smlp.predict_probs(texts, smlp.load(path), encoder).argmax(axis=1)
            models[fold] = path
            predictions[fold] = preds
            accuracies[fold] = accuracy_score(gold, preds)
            print(name, "fold", fold, "acc", round(accuracies[fold], 4))
        runs.append({'models': models, 'predictions': predictions, 'accuracies': accuracies})
    return runs

#systems = [
#   ('logistic_indicator-enc', token_indicator_encoder, 'logistic'),
#   ('logistic_bow-enc',       bag_of_words_encoder,    'logistic'),
#   ('mlp_indicator-enc',      token_indicator_encoder, 'mlp'),
#   ('mlp_bow-enc',            bag_of_words_encoder,    'mlp'),
#]
#for name, enc, kind in systems:
#    my_classifiers[name] = rebuild_system(name, enc, kind)
#   save_system(name)

#print(list(load_all_systems().keys()))